In [ ]:
# ==============================================================================
# The Climate Cost of Urban Sprawl within Exeter-Cranbrook Corridor
# Script 2: Validating Script 1 Sprawl Results with nDSM
# Date: April 2026
# ==============================================================================

In [ ]:
# Import required libraries
import os
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc

In [ ]:
# Setup Paths
# Local Users: Leave BASE_DIR = "" as it is
# Colab Users: Use your Drive path below (i.e BASE_DIR = "/content/drive/MyDrive/Folder_Name/")

BASE_DIR = ""

dtm_path = os.path.join(BASE_DIR, "Exeter_Cranbrook_DTM_1m.tif")
dsm_path = os.path.join(BASE_DIR, "Exeter_Cranbrook_DSM_1m.tif")
mask_path = os.path.join(BASE_DIR, "Validated_Sprawl_Mask_10m.tif") # Output from Script 1

ndsm_out_path = os.path.join(BASE_DIR, "Exeter_Cranbrook_nDSM_1m.tif")
stats_out_path = os.path.join(BASE_DIR, "Cranbrook_Validation_Stats.txt")
chart_out_path = os.path.join(BASE_DIR, "Structural_Validation.png")

print("PHASE 1: Calculating 1m Heights (nDSM)...")
with rasterio.open(dtm_path) as dtm_src, rasterio.open(dsm_path) as dsm_src:
    profile_1m = dtm_src.profile
    profile_1m.update(
    dtype=rasterio.float32,
    count=1,
    compress='lzw',
    nodata=-9999,
    tiled=True,
    blockxsize=256,
    blockysize=256
)

    # Read the DSM and DTM
    dtm_data = dtm_src.read(1, masked=True)
    dsm_data = dsm_src.read(1, masked=True)

    # Calculate nDSM securely, preserving true nodata holes
    ndsm_1m = np.ma.maximum(dsm_data - dtm_data, 0)

    # Save the 1m nDSM
    with rasterio.open(ndsm_out_path, 'w', **profile_1m) as dst:
        dst.write(ndsm_1m.filled(-9999).astype(np.float32), 1)

del dtm_data, dsm_data, ndsm_1m
gc.collect()
print("✅ Phase 1 Complete. RAM Cleared.")

print("PHASE 2: 10m Zonal Maximum Validation...")
with rasterio.open(mask_path) as mask_src, rasterio.open(ndsm_out_path) as ndsm_src:

    # Check CRS Alignment
    if mask_src.crs != ndsm_src.crs:
        raise ValueError(f"🚨 CRS MISMATCH! Mask: {mask_src.crs} | LiDAR: {ndsm_src.crs}")

    mask_10m = mask_src.read(1)
    ndsm_10m_max = np.full(mask_10m.shape, -9999, dtype=np.float32)

    # Reproject directly from the saved file on disk to save RAM
    reproject(
        source=rasterio.band(ndsm_src, 1),
        destination=ndsm_10m_max,
        src_transform=ndsm_src.transform,
        src_crs=ndsm_src.crs,
        dst_transform=mask_src.transform,
        dst_crs=mask_src.crs,
        resampling=Resampling.max
    )

    # Extract valid pixels, strictly ignoring nodata (-9999)
    valid_building_heights = ndsm_10m_max[(mask_10m > 0) & (ndsm_10m_max != -9999)]

del mask_10m, ndsm_10m_max
gc.collect()

print("PHASE 3: Generating Validation Chart & Stats...")
if len(valid_building_heights) > 0:
    median_h = np.median(valid_building_heights)
    hit_rate = (np.sum(valid_building_heights >= 2) / len(valid_building_heights)) * 100
else:
    median_h, hit_rate = 0, 0
    print("⚠️ Warning: No valid overlapping pixels found.")

stats_text = f"Median Block Height: {median_h:.2f}m\nStructural Hit Rate: {hit_rate:.1f}%"

# Save stats to a text file using the dynamic BASE_DIR variable
with open(stats_out_path, "w") as f:
    f.write(f"--- Cranbrook Structural Validation ---\n")
    f.write(f"Total Sprawl Pixels Evaluated: {len(valid_building_heights)}\n")
    f.write(f"Structural Threshold: >= 2.0 meters\n")
    f.write(f"{stats_text}\n")

# Generate Validation Chart
plt.figure(figsize=(12, 7))
sns.histplot(valid_building_heights, bins=40, kde=True, color='crimson', label='Maximum Height (per 10m block)')
plt.axvline(2, color='black', linestyle='--', linewidth=2, label='Structural Threshold (2m)')

plt.text(0.95, 0.85, stats_text, transform=plt.gca().transAxes, fontsize=12,
         fontweight='bold', ha='right', bbox=dict(facecolor='white', alpha=0.8, edgecolor='grey'))

plt.title('Structural Validation of Satellite Sprawl Detection', fontsize=14, fontweight='bold')
plt.xlabel('Maximum Vertical Height per 10m Pixel (m)', fontsize=12)
plt.ylabel('Frequency (10m Pixels)', fontsize=12)
plt.legend(loc='center right')

plt.tight_layout()
# Save the chart using the dynamic BASE_DIR variable
plt.savefig(chart_out_path, dpi=300)
plt.show()

print("✅ Analysis Complete. Chart, nDSM, and Text Report are ready.")